# 02 - LLM-as-judge baseline (Phase 3), GPU

**Settings:** Accelerator **GPU T4 x2** | Internet **On** | Input: your bundle dataset | Secret `HF_TOKEN` (recommended).
**Runtime:** ~5 min safety check, then ~1 h (four models, downloads included).

Judges the **same 1,628 held-out rows** as the probe and prints the probe's numbers next to them.

Protection against late failures:
* **STEP 3** checks every model is reachable *before* anything runs. A gated Llama repo that your token cannot open
  is **automatically replaced by an ungated mirror with the same weights** (it is labelled in the output).
* **STEP 4 preflight** runs the real pipeline on 6 rows (a few minutes). If it fails you find out right away.
* Each model runs in **its own process**; finished models are skipped on retry, a crash in one never affects the others,
  progress inside a model is checkpointed every 100 rows, and each model's download is deleted afterwards to save disk.
* **STEP 6** always runs and packages whatever finished. Results also sit in the Output tab under `results/llm_judge/`.

Tip: use **Save Version -> Save & Run All (Commit)** so it runs in the background.

In [ ]:
# STEP 1 - locate the uploaded bundle and load the helper module.
import os, sys
from pathlib import Path
_inp = Path(os.environ.get("LF_KAGGLE_INPUT", "/kaggle/input"))
_hits = sorted(_inp.rglob("kaggle_helpers.py"))
assert _hits, ("Bundle not found. In the right panel click 'Add Input' and add the Kaggle Dataset you created "
               "from linguafranca_bundle.zip, then run this cell again.")
sys.path.insert(0, str(_hits[0].parent))
import kaggle_helpers as kh
WORK = kh.setup()

In [ ]:
# STEP 2 - login, GPU check.
HF_TOKEN = kh.hf_login()
kh.require_gpu(min_total_gb=14)

In [ ]:
# STEP 3 - which models can run?
REQUESTED = (os.environ.get("LF_TEST_MODELS") or
             "meta-llama/Llama-3.2-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-7B-Instruct,meta-llama/Llama-3.1-8B-Instruct").split(",")
CHECK = kh.check_models(REQUESTED, HF_TOKEN)
USABLE = [m for m, repo, msg in CHECK if repo]
print("\nwill run:", USABLE)
assert USABLE, "No model is reachable. Check that Internet is ON, and read the messages above."

In [ ]:
# STEP 4 - PREFLIGHT: run the real script on 6 rows with the first model that works.
PRE = kh.RESULTS / "preflight_llm_judge"
passed = None
for m in USABLE:
    rc = kh.run([sys.executable, "evaluate_llm_baseline.py", "--data", "data/canonical/augmented_v2.jsonl",
                 "--models", m, "--max-items", "6", "--out", str(PRE), "--force", "--free-disk",
                 "--probe-predictions", "outputs/probing/test_predictions.csv"], "preflight_judge")
    if rc == 0 and list(PRE.glob("model_*.json")):
        passed = m; break
    print(f"[preflight] {m} did not pass; trying the next model")
assert passed, "PREFLIGHT FAILED for every model - see the log above (also in results/logs). Nothing long was started."
print(f"\nPREFLIGHT PASSED with {passed} - starting the main run.")

In [ ]:
# STEP 5 - MAIN RUN: one process per model (resumable; retried once if it dies).
OUT = kh.RESULTS / "llm_judge"
for m in USABLE:
    print("\n" + "#" * 90 + f"\n# {m}\n" + "#" * 90)
    kh.run_with_retries([sys.executable, "evaluate_llm_baseline.py", "--data", "data/canonical/augmented_v2.jsonl",
                         "--models", m, "--out", str(OUT), "--free-disk",
                         "--probe-predictions", "outputs/probing/test_predictions.csv"],
                        "judge_" + m.replace("/", "__"), attempts=2)

In [ ]:
# STEP 6 - FINAL (always runs): aggregate, show, package.
kh.run([sys.executable, "evaluate_llm_baseline.py", "--out", str(OUT), "--aggregate-only"], "aggregate")
kh.show(OUT / "summary.md")
kh.package(OUT, "llm_judge_results")
print("Download llm_judge_results.zip from the Output tab (or take files from results/llm_judge/).")